
# Edward Moswane Assignment 3: Python Refresher

#### **ATTENTION!!!**
- **Place each section (1,2...) in one or more cells to keep the notebook organised**
- **Complete the codes in the `TODO` sections**

In [66]:
# Python Library Management System

import pickle
import sqlite3
import requests

In [3]:
# 1. Data Types and Variables
book_title = "Python Programming"
book_quantity = 5
book_price = 29.99
is_available = True

print(f"Book: {book_title}, Quantity: {book_quantity}, Price: ${book_price}, Available: {is_available}")


Book: Python Programming, Quantity: 5, Price: $29.99, Available: True


In [48]:

# 2. Lists
books = ["Python Basics", "Advanced Python", "Data Science with Python"]

# Add two more books to the list
books.append("GEOLOGY")
books.insert(1, "MINING METHODS")

print("Updated book list:", books)

Updated book list: ['Python Basics', 'MINING METHODS', 'Advanced Python', 'Data Science with Python', 'GEOLOGY']


In [49]:
# 3. Loops and range()
print("\nAvailable books:")

# Using a for loop and range() to print books with their indices
for index in range(len(books)): print(f"Index {index}: {books[index]}")



Available books:
Index 0: Python Basics
Index 1: MINING METHODS
Index 2: Advanced Python
Index 3: Data Science with Python
Index 4: GEOLOGY


In [50]:
# 4. User Input
new_book = input("Enter a new book title: ")
books.append(new_book)
print("Updated book list:", books)

Enter a new book title: Economics for Mining
Updated book list: ['Python Basics', 'MINING METHODS', 'Advanced Python', 'Data Science with Python', 'GEOLOGY', 'Economics for Mining']


In [10]:
def check_availability(book_name):
  """ Check whether a book is available in the library.
  Returns:
   True if the book is in the list.
   False if the book is not in the list. """
  if not isinstance(book_name, str) or not book_name.strip():
     return False
  if book_name.strip().casefold() in [book.casefold() for book in books]:
      return True
  else: return False


In [11]:
# Test the function
print(f"Is 'Python Basics' available? {check_availability('Python Basics')}")
print( f"Is 'Nonexistent Book' available? " f"{check_availability('Nonexistent Book')}" )

Is 'Python Basics' available? True
Is 'Nonexistent Book' available? False


In [18]:
# 6. File I/O
def save_books_to_file(filename):
    with open(filename, 'w') as f:
        for book in books:
            f.write(f"{book}\n")

# Read books from a file
def read_books_from_file(filename):
    """
    Read books from a text file and return them as a list.
    """

    try:
        with open(filename, "r", encoding="utf-8") as f:
            return [line.strip() for line in f if line.strip()]

    except FileNotFoundError:
        print(f"File not found: {filename}")
        return []

    except OSError as error:
        print(f"Error reading books from file: {error}")
        return []



In [51]:
# Test file I/O

print("Books read from file:")
save_books_to_file("books.txt")
!cat books.txt

Books read from file:
Python Basics
MINING METHODS
Advanced Python
Data Science with Python
GEOLOGY
Economics for Mining


In [69]:
# 7. Pickle for Serialization
def save_books_pickle(filename):
    with open(filename, 'wb') as f:
        pickle.dump(books, f)

# Loading  books using Pickle
def load_books_pickle(filename):
    """
    Load the books list from a Pickle file.
    """

    try:
        with open(filename, "rb") as f:
            loaded_books = pickle.load(f)

        if isinstance(loaded_books, list):
            return loaded_books

        print("Pickle file does not contain a valid book list.")
        return []

    except FileNotFoundError:
        print(f"Pickle file not found: {filename}")
        return []

    except (OSError, pickle.PickleError, EOFError) as error:
        print(f"Error loading Pickle file: {error}")
        return []

In [71]:
# Test Pickle functions

# save the current books list to the Pickle file
save_books_pickle("books.pkl")

# load the books from the Pickle file
loaded_books = load_books_pickle("books.pkl")

print("Books loaded from pickle:", loaded_books)

Books loaded from pickle: ['Python Basics', 'MINING METHODS', 'Advanced Python', 'Data Science with Python', 'GEOLOGY', 'Economics for Mining']


In [42]:
# 8. SQLite Database Interaction
def create_books_table():
    conn = sqlite3.connect('library.db')
    cursor = conn.cursor()
    cursor.execute('''CREATE TABLE IF NOT EXISTS books
                      (id INTEGER PRIMARY KEY, title TEXT, quantity INTEGER)''')
    conn.commit()
    conn.close()

    def create_books_table():
     """
    Create the books table if it does not already exist.
    """

    try:
        conn = sqlite3.connect("library.db")
        cursor = conn.cursor()

        cursor.execute("""
            CREATE TABLE IF NOT EXISTS books
            (
                id INTEGER PRIMARY KEY,
                title TEXT NOT NULL,
                quantity INTEGER NOT NULL)
        """)

        conn.commit()
        conn.close()

        return True

    except sqlite3.Error as error:
        print(f"Error creating database table: {error}")
        return False

# Adding a book to the SQLite database
def add_book_to_db(title, quantity):
    """
    Add a book to the SQLite database.
    """

    # Checking for invalid title
    if not isinstance(title, str) or not title.strip():
        print("Error: Book title cannot be empty.")
        return False

    # Checking for invalid quantity
    if not isinstance(quantity, int) or quantity < 0:
        print("Error: Quantity must be a non-negative integer.")
        return False

    try:
        conn = sqlite3.connect("library.db")
        cursor = conn.cursor()

        cursor.execute("INSERT INTO books (title, quantity) VALUES (?, ?)",(title.strip(), quantity))

        conn.commit()
        conn.close()

        return True

    except sqlite3.Error as error:
        print(f"Error adding book to database: {error}")
        return False


In [43]:
# Test database functions
create_books_table()

True

In [45]:
# Test database functions
if create_books_table():

    if add_book_to_db("Python for Beginners", 10):
        print("Book successfully added to the SQLite database.")

    try:
        conn = sqlite3.connect("library.db")
        cursor = conn.cursor()

        cursor.execute("SELECT id, title, quantity FROM books")

        rows = cursor.fetchall()
        conn.close()

        print("\nBooks in SQLite database:")

        for row in rows:
            print(row)

    except sqlite3.Error as error:
        print(f"Error reading database: {error}")

Book successfully added to the SQLite database.

Books in SQLite database:
(1, 'Python for Beginners', 10)
(2, 'Python for Beginners', 10)
(3, 'Python for Beginners', 10)
(4, 'Python for Beginners', 10)


In [46]:
# 9. Web API Interaction (GitHub API)
def get_python_repos():
    url = "https://api.github.com/search/repositories"
    params = {"q": "language:python", "sort": "stars", "order": "desc"}
    response = requests.get(url, params=params)
    data = response.json()
    return [repo['name'] for repo in data['items'][:5]]

In [47]:
# Test API function
print("Top 5 Python repositories on GitHub:", get_python_repos())

print("\nLibrary Management System operations completed.")

Top 5 Python repositories on GitHub: ['public-apis', 'free-programming-books', 'system-design-primer', 'awesome-python', 'project-based-learning']

Library Management System operations completed.


Colab Link

In [ ]:
https://colab.research.google.com/drive/1MzjpAsGr0aIhKQOBcifQVWvAMx6xJFBg?usp=sharing

GitHUb